<a href="https://colab.research.google.com/github/Jujube470/3DS-FIDO/blob/main/myFIDO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Tuple, Any
import os, json, base64, hmac, hashlib, time, statistics
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey, Ed25519PublicKey


# ---------- utilities ----------
def b64e(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).decode().rstrip("=")

def b64d(s: str) -> bytes:
    pad = "=" * (-len(s) % 4)
    return base64.urlsafe_b64decode(s + pad)

def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def hmac_sha256(key: bytes, data: bytes) -> bytes:
    return hmac.new(key, data, hashlib.sha256).digest()

def hkdf32(ikm: bytes, salt: bytes, info: bytes) -> bytes:
    hkdf = HKDF(algorithm=hashes.SHA256(), length=32, salt=salt, info=info)
    return hkdf.derive(ikm)

def canon(obj: Any) -> bytes:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")

def now_ms() -> float:
    return time.perf_counter() * 1000.0


# ---------- protocol messages ----------
@dataclass
class RegChallenge:
    pan: str
    rs_b64: str
    ids: str

@dataclass
class AuthChallenge:
    pan: str
    rs_b64: str
    ids: str
    h_hex: str


# ---------- Authenticator ----------
class SoftwareAuthenticator:
    def __init__(self, master_sk_seed: bytes | None = None, seed_h: bytes | None = None):
        self.master_sk_seed = master_sk_seed or os.urandom(32)
        self.seed_h = seed_h or os.urandom(32)
        self.list_a: Dict[str, Tuple[bytes, int]] = {}  # h -> (sk_seed32, ctr_a)

    def _compute_h(self, pan: str) -> str:
        return hmac_sha256(self.seed_h, pan.encode()).hex()

    def _derive_sk_seed(self, pan: str, ids: str) -> bytes:
        salt = sha256((pan + "|" + ids).encode())
        return hkdf32(self.master_sk_seed, salt=salt, info=b"SK")

    def register(self, pan: str, rs_b64: str, ids: str) -> Tuple[dict, str]:
        h_hex = self._compute_h(pan)
        sk_seed = self._derive_sk_seed(pan, ids)
        sk = Ed25519PrivateKey.from_private_bytes(sk_seed)
        pk = sk.public_key().public_bytes_raw()

        ctr_a = int.from_bytes(os.urandom(8), "big")
        m = {
            "PAN": pan,
            "rs": rs_b64,
            "ids": ids,
            "h": h_hex,
            "pk": b64e(pk),
            "ctr_a": ctr_a,
        }
        sigma = sk.sign(canon(m))
        self.list_a[h_hex] = (sk_seed, ctr_a)
        return m, b64e(sigma)

    def authenticate(self, pan: str, rs_b64: str, ids: str, h_hex: str) -> Tuple[dict, str]:
        if h_hex not in self.list_a:
            raise ValueError("Unknown h (not registered on authenticator).")
        sk_seed, ctr_a = self.list_a[h_hex]
        ctr_a = ctr_a + 1
        sk = Ed25519PrivateKey.from_private_bytes(sk_seed)

        m = {
            "PAN": pan,
            "rs": rs_b64,
            "ids": ids,
            "h": h_hex,
            "ctr_a": ctr_a,
        }
        sigma = sk.sign(canon(m))
        self.list_a[h_hex] = (sk_seed, ctr_a)
        return m, b64e(sigma)


# ---------- RP Server ----------
class FidoRPServer:
    def __init__(self, rp_id: str):
        self.rp_id = rp_id
        self.list_r: Dict[str, Tuple[str, bytes, int]] = {}  # PAN -> (h, pk, ctr_r)
        self.pending_reg: Dict[str, str] = {}
        self.pending_auth: Dict[str, str] = {}

    def start_registration(self, pan: str) -> RegChallenge:
        rs_b64 = b64e(os.urandom(16))
        self.pending_reg[pan] = rs_b64
        return RegChallenge(pan=pan, rs_b64=rs_b64, ids=self.rp_id)

    def finish_registration(self, m: dict, sigma_b64: str) -> bool:
        pan = m["PAN"]
        if self.pending_reg.get(pan) != m["rs"]:
            return False
        if m["ids"] != self.rp_id:
            return False

        pk = b64d(m["pk"])
        sigma = b64d(sigma_b64)
        try:
            Ed25519PublicKey.from_public_bytes(pk).verify(sigma, canon(m))
        except Exception:
            return False

        # IMPORTANT: ctr_r sync with ctr_a at registration
        h_hex = m["h"]
        ctr_a = int(m["ctr_a"])
        self.list_r[pan] = (h_hex, pk, ctr_a)
        del self.pending_reg[pan]
        return True

    def start_authentication(self, pan: str) -> AuthChallenge:
        if pan not in self.list_r:
            raise ValueError("PAN not registered.")
        rs_b64 = b64e(os.urandom(16))
        self.pending_auth[pan] = rs_b64
        h_hex, _, _ = self.list_r[pan]
        return AuthChallenge(pan=pan, rs_b64=rs_b64, ids=self.rp_id, h_hex=h_hex)

    def finish_authentication(self, m: dict, sigma_b64: str) -> bool:
        pan = m["PAN"]
        if self.pending_auth.get(pan) != m["rs"]:
            return False
        if m["ids"] != self.rp_id:
            return False
        if pan not in self.list_r:
            return False

        h_hex, pk, ctr_r = self.list_r[pan]
        ctr_r = ctr_r + 1

        if m["h"] != h_hex:
            return False
        if int(m["ctr_a"]) != ctr_r:
            return False

        sigma = b64d(sigma_b64)
        try:
            Ed25519PublicKey.from_public_bytes(pk).verify(sigma, canon(m))
        except Exception:
            return False

        self.list_r[pan] = (h_hex, pk, ctr_r)
        del self.pending_auth[pan]
        return True


# ---------- Issuer Server ----------
class IssuerServer:
    def __init__(self, rp: FidoRPServer):
        self.rp = rp

    def start_registration(self, pan: str) -> RegChallenge:
        return self.rp.start_registration(pan)

    def finish_registration(self, m: dict, sigma_b64: str) -> bool:
        return self.rp.finish_registration(m, sigma_b64)

    def start_authentication(self, pan: str) -> AuthChallenge:
        return self.rp.start_authentication(pan)

    def finish_authentication(self, m: dict, sigma_b64: str) -> bool:
        return self.rp.finish_authentication(m, sigma_b64)


# ---------- FIDO Client (with timing) ----------
class FidoClient:
    def __init__(self, issuer: IssuerServer, authenticator: SoftwareAuthenticator, expected_rp_id: str):
        self.issuer = issuer
        self.authenticator = authenticator
        self.expected_rp_id = expected_rp_id

    def registration_flow_timed(self, pan: str) -> Tuple[bool, dict]:
        t0 = now_ms()
        # 1) Start registration (challenge)
        a0 = now_ms()
        ch = self.issuer.start_registration(pan)
        a1 = now_ms()

        if ch.ids != self.expected_rp_id:
            return False, {"total_ms": now_ms() - t0}

        # 2) Authenticator computes (m, sigma)
        b0 = now_ms()
        m, sig = self.authenticator.register(ch.pan, ch.rs_b64, ch.ids)
        b1 = now_ms()

        # 3) Finish registration (server verify/store)
        c0 = now_ms()
        ok = self.issuer.finish_registration(m, sig)
        c1 = now_ms()

        return ok, {
            "total_ms": now_ms() - t0,
            "start_ms": a1 - a0,
            "authenticator_ms": b1 - b0,
            "finish_ms": c1 - c0,
        }

    def authentication_flow_timed(self, pan: str) -> Tuple[bool, dict]:
        t0 = now_ms()
        # 1) Start auth (challenge)
        a0 = now_ms()
        ch = self.issuer.start_authentication(pan)
        a1 = now_ms()

        if ch.ids != self.expected_rp_id:
            return False, {"total_ms": now_ms() - t0}

        # 2) Authenticator computes (m, sigma)
        b0 = now_ms()
        m, sig = self.authenticator.authenticate(ch.pan, ch.rs_b64, ch.ids, ch.h_hex)
        b1 = now_ms()

        # 3) Finish auth (server verify/store)
        c0 = now_ms()
        ok = self.issuer.finish_authentication(m, sig)
        c1 = now_ms()

        return ok, {
            "total_ms": now_ms() - t0,
            "start_ms": a1 - a0,
            "authenticator_ms": b1 - b0,
            "finish_ms": c1 - c0,
        }


def summarize_ms(samples: list[float]) -> dict:
    samples_sorted = sorted(samples)
    return {
        "n": len(samples_sorted),
        "mean_ms": statistics.mean(samples_sorted),
        "median_ms": statistics.median(samples_sorted),
        "p95_ms": samples_sorted[int(0.95 * (len(samples_sorted) - 1))],
        "p99_ms": samples_sorted[int(0.99 * (len(samples_sorted) - 1))],
        "min_ms": samples_sorted[0],
        "max_ms": samples_sorted[-1],
    }


if __name__ == "__main__":
    rp = FidoRPServer(rp_id="rp.example.com")
    issuer = IssuerServer(rp=rp)
    auth = SoftwareAuthenticator()
    client = FidoClient(issuer=issuer, authenticator=auth, expected_rp_id="rp.example.com")

    PAN = "4111111111111111"  # demo only

    # --- Registration (usually once) ---
    ok_reg, reg_t = client.registration_flow_timed(PAN)
    print("registration ok:", ok_reg)
    print("registration timing (ms):", reg_t)

    # --- Authentication (run multiple times for stable stats) ---
    # warm-up
    for _ in range(10):
        client.authentication_flow_timed(PAN)

    N = 200
    totals, starts, auths, finishes = [], [], [], []
    oks = 0
    for _ in range(N):
        ok, t = client.authentication_flow_timed(PAN)
        oks += int(ok)
        totals.append(t["total_ms"])
        starts.append(t["start_ms"])
        auths.append(t["authenticator_ms"])
        finishes.append(t["finish_ms"])

    print(f"auth ok: {oks}/{N}")
    print("auth total:", summarize_ms(totals))
    print("auth start:", summarize_ms(starts))
    print("auth authenticator:", summarize_ms(auths))
    print("auth finish:", summarize_ms(finishes))

registration ok: True
registration timing (ms): {'total_ms': 0.7815670000127284, 'start_ms': 0.02907299999787938, 'authenticator_ms': 0.5181290000036824, 'finish_ms': 0.22948799999721814}
auth ok: 200/200
auth total: {'n': 200, 'mean_ms': 0.3528212199999689, 'median_ms': 0.3460839999970631, 'p95_ms': 0.39177699998253956, 'p99_ms': 0.4188139999896521, 'min_ms': 0.31245599999965634, 'max_ms': 0.46054000000003725}
auth start: {'n': 200, 'mean_ms': 0.014972640000341925, 'median_ms': 0.01502999999502208, 'p95_ms': 0.01910799997858703, 'p99_ms': 0.03405999999085907, 'min_ms': 0.010568000012426637, 'max_ms': 0.03435600000375416}
auth authenticator: {'n': 200, 'mean_ms': 0.14790896000100473, 'median_ms': 0.1456289999987348, 'p95_ms': 0.17560600000433624, 'p99_ms': 0.18750800000270829, 'min_ms': 0.12539400000241585, 'max_ms': 0.20335400001204107}
auth finish: {'n': 200, 'mean_ms': 0.18888370500040763, 'median_ms': 0.18479450000450015, 'p95_ms': 0.21368300000904128, 'p99_ms': 0.22680400000535883